# Movie Recommendations System

### Part 1 - Imports

In [95]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### Part 2 - Load and explore the data

In [96]:
dir = "ml-latest//"
movies = pd.read_csv(dir + "movies.csv")
ratings = pd.read_csv(dir + "ratings.csv", nrows =1000)
tags = pd.read_csv(dir + "tags.csv", nrows =1000)

#print(f"Movies : \n {movies.head()}")
#print(f"\n Ratings :\n {ratings.head()}")
#print(f"\n Tags : \n {tags.head()}")
movies.info()
# We check how many movies don't have a genre assigned
movies[movies["genres"] == "(no genres listed)"].shape[0]

# and now we remove those movies from the list as they won't be useful for the recommendation based on genres
movies = movies[movies["genres"] != "(no genres listed)"].reset_index(drop=True)
movies["genres"].shape[0]


n = 5 #This will be the number of suggestions we will see in the end result 
movie_title = "Moana" #This will be the movie that will compare with the rest

<class 'pandas.DataFrame'>
RangeIndex: 86537 entries, 0 to 86536
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  86537 non-null  int64
 1   title    86537 non-null  str  
 2   genres   86537 non-null  str  
dtypes: int64(1), str(2)
memory usage: 2.0 MB


### Clean the titles
I'll use regex to remove the years from the title and instead add them to a new column so they can be used for advanced filtering. 

In [97]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)$")
#This part extracted the part in parentheses containing 4 digits, and added it to a new column
movies["title"] = movies["title"].str.replace(r"\(\d{4}\)$", "", regex=True).str.strip()
#This line removes the years from the title with the same parameters as the previous
#regex=True tells pandas to interpret \(\d{4}\)$ as pattern instead of literal string of text
#str.strip() removes any leftover whitespace after removal

### Clean the movies genres

In [98]:
# since the genres in the database are divided by | and pandas expects a space we will adapt them
movies["genres"] = movies["genres"].str.replace("|", " ", regex=False)
movies["genres"].head()

0    Adventure Animation Children Comedy Fantasy
1                     Adventure Children Fantasy
2                                 Comedy Romance
3                           Comedy Drama Romance
4                                         Comedy
Name: genres, dtype: str

### Creating the TF-IDF vocabulary

In [99]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(movies["genres"])

print(tfidf_matrix.shape)

(79477, 21)


### Movie selection functions

In [100]:
def get_index (movie_title):
    result = movies[movies["title"]== movie_title]
    if result.empty:
        print(f"Movie {movie_title} not in the database")
        return None
    return result.index[0]

def get_vector(index, tfidf_matrix):
    row = tfidf_matrix[index].toarray()
    return row

def get_similarity (index, tfidf_matrix):
    row = get_vector(index, tfidf_matrix)
    similarity = cosine_similarity(row, tfidf_matrix)
    return similarity


### Recommend Movies

In [101]:
def recommend_movies(movie_title, tfidf_matrix):
    idx = get_index(movie_title)
    if idx is None:
        return

    similarity = get_similarity(idx, tfidf_matrix)
    sim = similarity[0]

    sorted_sim = sim.argsort()[::-1]
    topn = sorted_sim[1:n+1]

    matches = movies["title"].iloc[topn]
    return matches


### Running the model

In [103]:
print(recommend_movies(movie_title, tfidf_matrix))

79444                   Name Me Lawand
79441           Unknown: Killer Robots
106                            Catwalk
97       Heidi Fleiss: Hollywood Madam
126                     Jupiter's Wife
Name: title, dtype: str
